In [31]:
import pandas as pd
import os
import glob

In [32]:
ground_truth_df = pd.read_csv('../datasets/ground-truth.csv')
ground_truth_df

,Project,Antipattern,File,Line from,Line to,Comment
0,alexxiuccia/TrackMe,Rounding Errors,codice/TM_db/src-generated/it/trackme/jooq/gen...,71,71,toiteväärtuse andmed
1,alexxiuccia/TrackMe,Rounding Errors,codice/TM_db/src-generated/it/trackme/jooq/gen...,76,76,toiteväärtuse andmed
2,alexxiuccia/TrackMe,Rounding Errors,codice/TM_db/src-generated/it/trackme/jooq/gen...,81,81,toiteväärtuse andmed
3,alexxiuccia/TrackMe,Keyless Entry,codice/TM_db/src-generated/it/trackme/jooq/gen...,55,55,viide ricetta tabeli primaarvõtmele
4,alexxiuccia/TrackMe,Keyless Entry,codice/TM_db/src-generated/it/trackme/jooq/gen...,60,60,viide alimento tabeli primaarvõtmele
...,...,...,...,...,...,...
1557,ydeng11/Minance,Implicit Columns,src/main/java/today/ihelio/minance/service/Ban...,61,61,select(TABLE).from(TABLE)
1558,ydeng11/Minance,Implicit Columns,src/main/java/today/ihelio/minance/service/Cat...,115,115,selectDistinct(TABLE).from(TABLE)
1559,ydeng11/Minance,Implicit Columns,src/main/java/today/ihelio/minance/service/Tra...,98,98,select(TABLE).from(TABLE)
1560,ydeng11/Minance,Implicit Columns,src/main/java/today/ihelio/minance/service/Tra...,105,105,select(TABLE).from(TABLE)


In [33]:
files_with_antipatterns_df = ground_truth_df[["Project", "File"]].drop_duplicates()
files_with_antipatterns_df

,Project,File
0,alexxiuccia/TrackMe,codice/TM_db/src-generated/it/trackme/jooq/gen...
3,alexxiuccia/TrackMe,codice/TM_db/src-generated/it/trackme/jooq/gen...
5,alexxiuccia/TrackMe,codice/TM_db/src-generated/it/trackme/jooq/gen...
7,alexxiuccia/TrackMe,codice/TM_db/src-generated/it/trackme/jooq/gen...
10,alexxiuccia/TrackMe,codice/TM_db/src-generated/it/trackme/jooq/gen...
...,...,...
1549,ydeng11/Minance,src/main/java/today/ihelio/jooq/tables/Transac...
1550,ydeng11/Minance,src/main/java/today/ihelio/minance/service/Acc...
1554,ydeng11/Minance,src/main/java/today/ihelio/minance/service/Ban...
1558,ydeng11/Minance,src/main/java/today/ihelio/minance/service/Cat...


In [34]:
annotated_projects_df = ground_truth_df[["Project"]].drop_duplicates()
annotated_projects_df

,Project
0,alexxiuccia/TrackMe
18,alvin-qh/study-java
52,Andrey582/NotificationBot
71,anyamilichenko/soa
91,APSfurizon/fz-backend
...,...
1315,therepanic/trustwin-casino-project
1420,wuhunyu/jooq-transaction
1423,xnelo/filearch
1439,yangjinguang/i-share-server


In [35]:
# Excluding files generated based on system schemas and database extensions/migration tools
excluded_path_fragments = [
    # Java
    'module-info.java',
    'package-info.java',
    
    # MySQL
    'information_schema/InformationSchema.java',
    'information_schema/tables',
    'mysql/Mysql.java',
    'mysql/tables',
    'performance_schema/PerformanceSchema.java',
    'performance_schema/tables',
    'sys/Sys.java',
    'sys/tables',

    # Flyway
    'FlywaySchemaHistory.java',

    # Liquibase
    'Databasechangelog.java',
    'Databasechangeloglock.java',

    # PostgreSQL
    'information_schema/Domains.java',
    'pg_catalog/PgCatalog.java',
    'pg_catalog/tables',

    # Quartz
    'tables/Qrtz',

    # PGP
    'PgpArmorHeaders.java',

    # Dblink
    'tables/Dblink',

    # hstore
    'tables/Each.java',
    'tables/Skeys.java',
    'tables/Svals.java'
]

# Including files that contain a reference to jOOQ or a potential reference to a generated file (the latter may include false positives)
included_phrases = [
    'org.jooq',
    'Tables',
    'tables'
]

# Excluding generated classes, which contain redundant information that doesn't help with detection
excluded_phrases = [
    'extends TableRecordImpl',
    'extends UpdatableRecordImpl',
    'extends org.jooq.impl.UpdatableRecordImpl',
    'extends CatalogImpl',
    'extends SchemaImpl',
    'extends UDTImpl',
    'extends UDTRecordImpl',
    'extends DAOImpl',
    'extends AbstractRoutine',
    'implements EnumType',
    'implements DeserializableJooqEnum',
    'public class Indexes',
    'public class Keys',
    'public class Routines',
    'public class Sequences',
    'public class Tables',
    'public class Domains',
    'class AbstractSpringDAOImpl',
    'extends AbstractSpringDAOImpl',
    'tables.pojos;',
    'tables.interfaces;',
    'Targeted by JavaCPP'
]

In [39]:
files_df = pd.DataFrame({
    "Project": pd.Series(dtype="str"),
    "File": pd.Series(dtype="str")
})
files_df

,Project,File


In [40]:
for index, row in annotated_projects_df.iterrows():
    project_dir = f"../repositories/{row['Project'].replace('/', '_')}"
    all_java_file_paths = glob.glob(project_dir + "/**/src/**/*.java", recursive=True)
    count = 0
    for file_path in all_java_file_paths:
        if all(fragment not in file_path for fragment in excluded_path_fragments) and os.path.isfile(file_path):
            with open(file_path, encoding='latin-1') as file:
                file_contents = file.read()
                if any(phrase in file_contents for phrase in included_phrases) and all(phrase not in file_contents for phrase in excluded_phrases):
                    files_df.loc[len(files_df)] = [row['Project'], file_path[17 + len(row['Project']):]]
files_df

,Project,File
0,alexxiuccia/TrackMe,codice/TM_db/src/main/java/it/trackme/TM_db/Ap...
1,alexxiuccia/TrackMe,codice/TM_db/src/main/java/it/trackme/TM_db/Ge...
2,alexxiuccia/TrackMe,codice/TM_fxml/src/main/java/it/trackme/TM_fxm...
3,alexxiuccia/TrackMe,codice/TM_logic/src/main/java/it/trackme/TM_lo...
4,alexxiuccia/TrackMe,codice/TM_logic/src/main/java/it/trackme/TM_lo...
...,...,...
1816,ydeng11/Minance,src/test/java/today/ihelio/minance/rest/Overvi...
1817,ydeng11/Minance,src/test/java/today/ihelio/minance/service/Acc...
1818,ydeng11/Minance,src/test/java/today/ihelio/minance/service/Ban...
1819,ydeng11/Minance,src/test/java/today/ihelio/minance/service/Cat...


In [41]:
comparison = pd.merge(files_df, files_with_antipatterns_df, how="left", indicator=True)
files_without_antipatterns_df = comparison[comparison["_merge"] == "left_only"].drop(columns="_merge")
files_without_antipatterns_df

,Project,File
0,alexxiuccia/TrackMe,codice/TM_db/src/main/java/it/trackme/TM_db/Ap...
1,alexxiuccia/TrackMe,codice/TM_db/src/main/java/it/trackme/TM_db/Ge...
2,alexxiuccia/TrackMe,codice/TM_fxml/src/main/java/it/trackme/TM_fxm...
4,alexxiuccia/TrackMe,codice/TM_logic/src/main/java/it/trackme/TM_lo...
5,alexxiuccia/TrackMe,codice/TM_logic/src/main/java/it/trackme/TM_lo...
...,...,...
1816,ydeng11/Minance,src/test/java/today/ihelio/minance/rest/Overvi...
1817,ydeng11/Minance,src/test/java/today/ihelio/minance/service/Acc...
1818,ydeng11/Minance,src/test/java/today/ihelio/minance/service/Ban...
1819,ydeng11/Minance,src/test/java/today/ihelio/minance/service/Cat...


In [51]:
seed = 123456
sample_percentage = 0.1

annotated_sample = files_with_antipatterns_df.sample(frac=sample_percentage, random_state=seed)
annotated_sample_file_count = len(annotated_sample)

unannotated_sample = files_without_antipatterns_df.sample(n=annotated_sample_file_count, random_state=seed)

sample = pd.concat([annotated_sample, unannotated_sample]).sort_values(by=["Project", "File"], ignore_index=True)
sample

,Project,File
0,APSfurizon/fz-backend,jooq-common/src/main/java/net/furizon/jooq/gen...
1,APSfurizon/fz-backend,jooq-common/src/main/java/net/furizon/jooq/gen...
2,BuchnevDmitry/LinkTracker,scrapper/src/main/java/edu/java/scrapper/domai...
3,Dankoy/jforwarder,telegram_chat_service/src/main/java/ru/dankoy/...
4,DiSSCo/dissco-core-annotation-processing-service,src/main/java/eu/dissco/annotationprocessingse...
...,...,...
123,therepanic/trustwin-casino-project,management-game-service/src/main/java/eu/panic...
124,wuhunyu/jooq-transaction,src/test/java/top/wuhunyu/jooq/transaction/ser...
125,yangjinguang/i-share-server,src/main/java/com/pingyueryou/ishare/jooq/tabl...
126,yangjinguang/i-share-server,src/main/java/com/pingyueryou/ishare/jooq/tabl...


In [52]:
sample.to_csv('../datasets/files-to-reannotate.csv', index=False)